# Lab · Bandit oder A/B-Test? — Starter-Notebook

**AKE · Gruppenvortrag „Vom Modell in den Betrieb“ · 26.11.2026** · Bezug: Folien Teil 3 (A/B-Testing) und Teil 4 (Bandits), Woche 5 (ε-greedy)

> **Dieses Notebook ist das komplette Lab.** Aufgaben, Erwartungswerte und Leitfragen stehen direkt hier. Die Kommentare im Code
> (Zeilen mit `#`) erklären Zeile für Zeile, was passiert.

**Fall Feldwerk (fiktiv):** Das Prüfmodell schreibt zu jedem Teil einen Befundtext; es gibt drei Prompt-Varianten **A, B, C**.
Erfolg = der Werker übernimmt den Befund **ohne Korrektur**. Wahre Übernahmequoten (kennt nur die Simulation): A 0,60 · B 0,65 · C 0,72.
Sie vergleichen einen **klassischen A/B-Test**, **ε-Greedy** und **Thompson Sampling** über 3.000 Befunde und messen den **Regret**
(≈ Zahl der unnötigen Korrekturen gegenüber „immer C“).

**Ihre Zeit ≈ 10 Min.** · Rechenzeit ≈ 5 s · Umgebung: nur `numpy`, `scipy`, `pandas`, `matplotlib`, kein neues Paket

| Aufgabe | Inhalt | Zeit |
|---|---|---|
| 0 · Setup | `AUSFÜHREN` | 1 Min. |
| 1 · A/B-Test | Fallzahl und Signifikanztest `AUSFÜHREN`, Gewinner wählen `CODE` | 3 Min. |
| 2 · Bandits | ε-Greedy `CODE`, Thompson Sampling `CODE` | 3 Min. |
| 3 · Vergleich | Tabelle, Grafik, Experiment `AUSFÜHREN` + `DENKEN` | 3 Min. |

| Marke | Was Sie tun |
|---|---|
| `CODE` | eine Lücke füllen: Zeile mit `raise NotImplementedError` durch Ihre Lösung ersetzen (Hinweis steht im Kommentar darüber) |
| `AUSFÜHREN` | nur ausführen und die Ausgabe lesen |
| `DENKEN` | Frage in der Markdown-Zelle beantworten |

Nach jeder Aufgabe zeigt eine Prüfung ✅ oder ❌ mit Hinweis. Wer länger als 2 Min. an einer Lücke hängt, übernimmt die Zeile aus der Lösung am Beamer.

## Aufgabe 0 · Setup

`AUSFÜHREN` · Bibliotheken, Prüfhelfer und die Simulation. Wenn alle Zeilen ✅ zeigen, kann es losgehen.

In [ ]:
import numpy as np                                      # Rechnen mit Vektoren und Matrizen
import pandas as pd                                     # Tabellen
import matplotlib.pyplot as plt                         # Diagramme
from scipy import stats                                 # Normalverteilung (Fallzahl), Chi-Quadrat-Test

# --- Pruefhelfer: check("Name", Bedingung) -> ✅ oder ❌ mit Hinweis (Bedingung als lambda, damit Fehler nicht abbrechen)
CHECKS = {}
def check(name, ok, hint=""):
    try:
        ok = bool(ok() if callable(ok) else ok); msg = hint
    except NotImplementedError as e:                    # die Luecke ist noch nicht gefuellt
        ok, msg = False, f"{e} ist noch offen"
    except Exception as e:                              # anderer Fehler in der Loesung
        ok, msg = False, f"{type(e).__name__}: {e}"
    CHECKS[name] = ok
    print(f"✅ {name}" if ok else f"❌ {name} – Hinweis: {msg}")

def nah(a, b, tol):                                     # |a - b| <= tol (Kontrollwerte mit Rundung)
    return abs(float(a) - float(b)) <= tol

# --- Der Fall Feldwerk ----------------------------------------------------------------------------------------
P = np.array([0.60, 0.65, 0.72])     # wahre Uebernahmequoten A, B, C – nur die Simulation kennt sie
T, R = 3000, 500                     # 3.000 Befunde je Durchlauf, 500 Durchlaeufe gleichzeitig (je eine Zeile)
BESTE = P.max()                      # 0,72: daran wird der Regret gemessen

def simuliere(strategie, seed=0, **parameter):
    """Spielt T Befunde in R Durchlaeufen. Rueckgabe: kumulierter Regret, Anteil der Wahl C je Schritt, Merkzettel."""
    rng = np.random.default_rng(seed)                   # fester Seed -> immer dasselbe Ergebnis
    erfolge, versuche = np.zeros((R, 3)), np.zeros((R, 3))   # S und N je Durchlauf und Variante: alles, was eine Strategie weiss
    zeilen, regret, anteil_c, merker = np.arange(R), np.zeros(T), np.zeros(T), {}
    for t in range(T):                                  # ein Befund nach dem anderen
        wahl = strategie(t, erfolge, versuche, rng, merker, **parameter)   # R Zahlen aus {0 = A, 1 = B, 2 = C}
        uebernommen = rng.random(R) < P[wahl]           # wuerfeln mit der wahren Quote der gewaehlten Variante
        erfolge[zeilen, wahl] += uebernommen            # die Strategie lernt nur aus diesen Zaehlern
        versuche[zeilen, wahl] += 1
        regret[t] = np.mean(BESTE - P[wahl])            # verschenkter Erfolg in diesem Schritt (z. B. B statt C: 0,07)
        anteil_c[t] = np.mean(wahl == 2)
    return np.cumsum(regret), anteil_c, merker

check("Simulation laeuft (Probe 'immer C' hat Regret 0)", lambda: simuliere(lambda t, *a: np.full(R, 2))[0][-1] == 0)

## Aufgabe 1 · Der klassische A/B-Test

**1.1 Fallzahl** `AUSFÜHREN` · Formel wie auf der Folie „A/B-Test richtig planen und auswerten“: geplant mit ≈ 65 % und kleinstem relevantem Unterschied
7 Prozentpunkte, α = 5 %, Power = 80 %. **Erwartung:** 691 je Variante → Testphase 2.073 von 3.000 Befunden.

**1.2 Gewinner wählen** `CODE` (1 Zeile) · In der Testphase kommen A, B, C reihum dran. Danach entscheidet der Test **einmal**: Je
Durchlauf gewinnt die Variante mit der höchsten gemessenen Quote $Q_a = S_a / N_a$ = `erfolge / versuche` – dieselbe Regel
$a = \arg\max_a Q_a$ wie der „gierige“ Teil von ε-Greedy auf **Folie 38**. Hilfe: `np.argmax(matrix, axis=1)` liefert je Zeile die Spalte mit dem größten Wert.
**Erwartung:** Regret **131,7** · richtiger Gewinner (C) in **99,4 %** der Durchläufe.

In [ ]:
# --- 1.1 Fallzahl (Formel von Folie 25) ----------------------------------------------------------------------
def fallzahl(p1, p2, alpha=0.05, power=0.80):
    z_a, z_b = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)        # 1,96 und 0,84
    p_m = (p1 + p2) / 2
    return int(np.ceil((z_a * np.sqrt(2 * p_m * (1 - p_m)) + z_b * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2 / (p1 - p2) ** 2))
N_TEST = fallzahl(0.65, 0.72)
print(f"Fallzahl: {N_TEST} je Variante -> Testphase {3 * N_TEST} von {T} Befunden")

# --- 1.2 Strategie A/B-Test -------------------------------------------------------------------------------------
def ab_test(t, erfolge, versuche, rng, merker, n_pro_variante):
    if t < 3 * n_pro_variante:                          # Testphase: Befund 0 -> A, 1 -> B, 2 -> C, 3 -> A, ...
        return np.full(R, t % 3)
    if "gewinner" not in merker:                        # erster Schritt nach der Testphase: einmal entscheiden
        # TODO 1: merker["gewinner"] = argmax_a Q_a mit Q_a = S_a / N_a (Folie 38) -> np.argmax(erfolge / versuche, axis=1)
        raise NotImplementedError("TODO 1")
    return merker["gewinner"]                           # danach nur noch der Gewinner

reg_ab, anteil_ab, merk_ab = simuliere(ab_test, n_pro_variante=N_TEST)
richtig_ab = np.mean(merk_ab["gewinner"] == 2)          # Anteil der Durchlaeufe, in denen C gewonnen hat
print(f"A/B-Test: Regret {reg_ab[-1]:.1f} | richtiger Gewinner in {richtig_ab:.1%}")

# --- Zum Vergleich: so wird ein einzelner Test ausgewertet (Chi-Quadrat-Test ueber drei Varianten) ------------------
u = np.random.default_rng(42).binomial(N_TEST, P)       # uebernommene Befunde je Variante in einem Test
p_wert = stats.chi2_contingency(np.column_stack([u, N_TEST - u]))[1]
print(f"Ein einzelner Test: A {u[0] / N_TEST:.1%} · B {u[1] / N_TEST:.1%} · C {u[2] / N_TEST:.1%} -> p = {p_wert:.1e} (signifikant)")

check("1.1 Fallzahl 691 je Variante", N_TEST == 691)
check("1.2 A/B-Test: Regret 131,7 und 99,4 % richtig", lambda: nah(reg_ab[-1], 131.7, 0.2) and nah(richtig_ab, 0.994, 0.002))

## Aufgabe 2 · Bandits

**2.1 ε-Greedy** `CODE` (1 Zeile) · Mit Wahrscheinlichkeit ε zufällig **erkunden**, sonst die bisher beste Variante **ausnutzen**
(wie die ε-greedy-Politik des DQN aus Woche 5). Formel von **Folie 38**: $a_t = \arg\max_a Q_a$ mit Wahrscheinlichkeit $1 - \varepsilon$, sonst zufällig.
`beste` ($\arg\max_a Q_a$) und `zufall` sind schon berechnet. Ergänzen Sie die Rückgabe mit `np.where(bedingung, wenn_wahr, sonst)`.
**Erwartung (ε = 0,1):** Regret **41,9**.

**2.2 Thompson Sampling** `CODE` (1 Zeile) · Formel von **Folie 38**: $\theta_a \sim \mathrm{Beta}(S_a + 1,\ F_a + 1)$, das größte gezogene $\theta$ gewinnt.
Im Code: $S_a$ = `erfolge`, $F_a$ = Misserfolge = `versuche - erfolge`. Die Beta-Verteilung beschreibt, wo die wahre Quote plausibel liegt. Hilfe: `rng.beta(a, b)`.
**Erwartung:** Regret **28,9**.

In [ ]:
# --- 2.1 epsilon-Greedy ----------------------------------------------------------------------------------------
def eps_greedy(t, erfolge, versuche, rng, merker, eps=0.1):
    quote = np.where(versuche > 0, erfolge / np.maximum(versuche, 1), np.inf)   # bisherige Quote; nie gespielt -> unendlich
    beste = np.argmax(quote + rng.random(quote.shape) * 1e-9, axis=1)           # beste Variante (Gleichstand zufaellig)
    zufall = rng.random(R) < eps                                                # in welchen Durchlaeufen wird erkundet?
    # TODO 2.1: Folie 38: mit Wkt. 1 - eps argmax_a Q_a (= beste), sonst zufaellig (rng.integers(0, 3, R)) -> np.where(zufall, ..., beste)
    raise NotImplementedError("TODO 2.1")

# --- 2.2 Thompson Sampling -------------------------------------------------------------------------------------
def thompson(t, erfolge, versuche, rng, merker):
    # TODO 2.2: Folie 38: theta_a ~ Beta(S_a + 1, F_a + 1) mit S_a = erfolge, F_a = versuche - erfolge -> zug = rng.beta(...)
    raise NotImplementedError("TODO 2.2")
    return np.argmax(zug, axis=1)                       # Variante mit dem hoechsten Zug gewinnt

reg_eps, anteil_eps, _ = simuliere(eps_greedy, eps=0.1)
reg_ts, anteil_ts, _ = simuliere(thompson)
print(f"ε-Greedy: Regret {reg_eps[-1]:.1f} | Thompson Sampling: Regret {reg_ts[-1]:.1f}")
check("2.1 ε-Greedy: Regret 41,9", lambda: nah(reg_eps[-1], 41.9, 0.2))
check("2.2 Thompson Sampling: Regret 28,9", lambda: nah(reg_ts[-1], 28.9, 0.2))

## Aufgabe 3 · Vergleich

`AUSFÜHREN` · Tabelle, Grafik und ein kurzes Experiment: Was passiert, wenn man den A/B-Test **ohne Fallzahlrechnung abkürzt**
(100 statt 691 Befunde je Variante)? **Erwartung:** Thompson spart **78 %** gegenüber dem A/B-Test ·
abgekürzter Test: Regret ≈ 62,7, aber richtiger Gewinner nur in **80 %**.

In [ ]:
ERG = {"A/B-Test": (reg_ab, anteil_ab), "ε-Greedy (ε = 0,1)": (reg_eps, anteil_eps), "Thompson Sampling": (reg_ts, anteil_ts)}
display(pd.DataFrame([{"Strategie": k, "Regret (unnoetige Korrekturen)": round(r[-1], 1),
                       "Anteil C letzte 300": f"{a[-300:].mean():.0%}"} for k, (r, a) in ERG.items()]).set_index("Strategie"))

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for k, (r, a) in ERG.items():
    ax[0].plot(r, label=k); ax[1].plot(pd.Series(a).rolling(100).mean(), label=k)
ax[0].set(title="Kumulierter Regret", xlabel="Befunde", ylabel="unnoetige Korrekturen"); ax[0].legend()
ax[1].set(title="Anteil der besten Variante C", xlabel="Befunde", ylim=(0, 1))
ax[1].axvline(3 * N_TEST, color="grey", ls="--", lw=1); ax[1].text(3 * N_TEST + 30, 0.05, "Ende Testphase", color="grey")
plt.tight_layout(); plt.show()

ersparnis = 1 - reg_ts[-1] / reg_ab[-1]
r_kurz, _, m_kurz = simuliere(ab_test, n_pro_variante=100)           # Experiment: Testphase ohne Fallzahl abgekuerzt
richtig_kurz = np.mean(m_kurz["gewinner"] == 2)
print(f"Thompson spart {ersparnis:.0%} gegenueber dem A/B-Test.")
print(f"Abgekuerzter A/B-Test (100 je Variante): Regret {r_kurz[-1]:.1f}, aber richtiger Gewinner nur in {richtig_kurz:.0%}")
check("3 Thompson spart rund 78 %", lambda: nah(ersparnis, 0.78, 0.01))
check("3 abgekuerzter Test waehlt nur in ≈ 80 % richtig", lambda: nah(richtig_kurz, 0.80, 0.01))
print(f"\nPruefungen: {sum(CHECKS.values())} ✅ / {sum(not v for v in CHECKS.values())} ❌")

**`DENKEN`**

a) Warum steigt die Regret-Linie des A/B-Tests bis Befund 2.073 gerade an und ist danach flach – und warum knicken die Bandits so früh ab?
b) Der abgekürzte Test verschenkt weniger. Warum ist er trotzdem keine gute Idee?
c) Feldwerk will von v1 auf v2 umsteigen **und** die Prompts optimieren: welches Verfahren wofür?

**Meine Antworten:** *…*